# 🔬 DiffPool GNN — 3-D Graph Evolution Visualizer



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import TUDataset
from torch_geometric.nn import DenseGCNConv, dense_diff_pool
from torch_geometric.utils import to_dense_adj, to_dense_batch
from torch_geometric.data import Batch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore")

print("All imports OK")
print(f"PyTorch {torch.__version__} | device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


All imports OK
PyTorch 2.10.0 | device: cpu


## 1 · Hyperparameters
Edit the values below — this replaces the Streamlit sidebar sliders.


In [2]:
# ╔══════════════════════════════════════╗
# ║      HYPERPARAMETER SETTINGS         ║
# ╚══════════════════════════════════════╝

EPOCHS     = 120          # 20 – 300
LR         = 1e-3         # 5e-4 | 1e-3 | 2e-3 | 5e-3
BATCH_SIZE = 32           # 8 | 16 | 32 | 64
HIDDEN     = 64           # 32 | 64 | 128
MAX_NODES  = 100          # max graph size to include (50 – 200)

# ── display summary ──
print("┌─────────────────────────────────┐")
print("│  Hyperparameter summary          │")
print("├─────────────────────────────────┤")
for k, v in [("Epochs", EPOCHS), ("Learning rate", LR), ("Batch size", BATCH_SIZE),
              ("Hidden dim", HIDDEN), ("Max nodes", MAX_NODES)]:
    print(f"│  {k:<18} {str(v):<12} │")
print("└─────────────────────────────────┘")


┌─────────────────────────────────┐
│  Hyperparameter summary          │
├─────────────────────────────────┤
│  Epochs             120          │
│  Learning rate      0.001        │
│  Batch size         32           │
│  Hidden dim         64           │
│  Max nodes          100          │
└─────────────────────────────────┘


## 2 · Model definition

In [3]:
class GNNBlock(nn.Module):
    """Dense GCN block used in DiffPool (embed + pool heads)."""
    def __init__(self, in_ch, hid_ch, out_ch):
        super().__init__()
        self.conv1 = DenseGCNConv(in_ch, hid_ch)
        self.conv2 = DenseGCNConv(hid_ch, hid_ch)
        self.conv3 = DenseGCNConv(hid_ch, out_ch)
        self.bn1 = nn.BatchNorm1d(hid_ch)
        self.bn2 = nn.BatchNorm1d(hid_ch)
        self.bn3 = nn.BatchNorm1d(out_ch)

    def forward(self, x, adj, mask=None):
        B, N, _ = x.size()
        def apply_bn(bn, t):
            return bn(t.view(B*N, -1)).view(B, N, -1)
        x = F.selu(apply_bn(self.bn1, self.conv1(x, adj, mask)))
        x = F.selu(apply_bn(self.bn2, self.conv2(x, adj, mask)))
        x = F.selu(apply_bn(self.bn3, self.conv3(x, adj, mask)))
        return x


class DiffPoolNet(nn.Module):
    """2-level DiffPool network for graph classification."""
    def __init__(self, in_features, num_classes, max_nodes=100, hidden=64):
        super().__init__()
        p1 = max(10, max_nodes // 4)
        p2 = max(4,  p1 // 4)
        self.p1, self.p2 = p1, p2
        self.embed1 = GNNBlock(in_features, hidden, hidden)
        self.pool1  = GNNBlock(in_features, hidden, p1)
        self.embed2 = GNNBlock(hidden, hidden, hidden)
        self.pool2  = GNNBlock(hidden, hidden, p2)
        self.embed3 = GNNBlock(hidden, hidden, hidden)
        self.cls = nn.Sequential(
            nn.Linear(hidden, hidden), nn.SELU(), nn.Dropout(0.45),
            nn.Linear(hidden, hidden//2), nn.SELU(), nn.Dropout(0.25),
            nn.Linear(hidden//2, num_classes),
        )

    def forward(self, x, adj, mask=None, return_intermediates=False):
        s1 = self.pool1(x, adj, mask)
        h1 = self.embed1(x, adj, mask)
        x1, adj1, lp1, le1 = dense_diff_pool(h1, adj, s1, mask)
        s2 = self.pool2(x1, adj1)
        h2 = self.embed2(x1, adj1)
        x2, adj2, lp2, le2 = dense_diff_pool(h2, adj1, s2)
        h3   = self.embed3(x2, adj2)
        read = h3.mean(dim=1)
        logits = self.cls(read)
        aux = (lp1 + le1 + lp2 + le2).mean()
        if return_intermediates:
            return F.log_softmax(logits, dim=-1), aux, {
                "x0": x,  "adj0": adj,
                "x1": x1, "adj1": adj1, "s1": s1,
                "x2": x2, "adj2": adj2, "s2": s2,
            }
        return F.log_softmax(logits, dim=-1), aux

print("Model classes defined.")


Model classes defined.


## 3 · Data loading

In [4]:
def load_proteins(max_nodes=100):
    dataset = TUDataset(root="data/TUDataset", name="PROTEINS")
    filtered = [d for d in dataset if d.num_nodes <= max_nodes]
    torch.manual_seed(42)
    perm    = torch.randperm(len(filtered))
    n_train = int(0.8 * len(filtered))
    train_data = [filtered[i] for i in perm[:n_train]]
    test_data  = [filtered[i] for i in perm[n_train:]]
    actual_max = max(d.num_nodes for d in filtered)
    n_feat     = filtered[0].num_node_features
    n_cls      = dataset.num_classes
    return train_data, test_data, actual_max, n_feat, n_cls, filtered


def dense_batch(data_list, max_nodes, device):
    batch = Batch.from_data_list(data_list)
    x, mask = to_dense_batch(batch.x, batch.batch, max_num_nodes=max_nodes)
    adj = to_dense_adj(batch.edge_index, batch.batch, max_num_nodes=max_nodes)
    return x.to(device), adj.to(device), mask.to(device), batch.y.to(device)


print("Loading PROTEINS dataset …")
train_data, test_data, ACT_MAX, N_FEAT, N_CLS, all_data = load_proteins(MAX_NODES)
print(f"✅ Loaded — {len(train_data)} train / {len(test_data)} test | "
      f"{N_FEAT} features | {N_CLS} classes | max nodes {ACT_MAX}")


Loading PROTEINS dataset …
✅ Loaded — 836 train / 210 test | 3 features | 2 classes | max nodes 100


## 4 · Visualization helpers (shared)

In [5]:
BLK      = "#000000"
DARK_BG  = "#ffffff"
PANEL_BG = "#f8fafc"
GRID_COL = "#d1d5db"

PLOTLY_LAYOUT = dict(
    paper_bgcolor=DARK_BG,
    plot_bgcolor=PANEL_BG,
    font=dict(color=BLK, size=11, family="Arial, sans-serif"),
    margin=dict(l=10, r=10, t=45, b=10),
    title_font=dict(color=BLK, size=13),
)

def _ax2d(title=""):
    return dict(
        title=dict(text=title, font=dict(color=BLK, size=12)),
        color=BLK, tickfont=dict(color=BLK, size=11),
        gridcolor=GRID_COL, linecolor=BLK, zerolinecolor=GRID_COL,
    )

def _ax3d(title="", show_ticks=True):
    return dict(
        title=dict(text=title, font=dict(color=BLK, size=12)),
        tickfont=dict(color=BLK, size=10),
        showticklabels=show_ticks, showgrid=True,
        gridcolor=GRID_COL, showline=True,
        linecolor=BLK, zerolinecolor=GRID_COL,
    )

def _colorbar(title=""):
    return dict(
        thickness=10,
        title=dict(text=title, font=dict(color=BLK, size=11)) if title else None,
        tickfont=dict(color=BLK, size=10),
        outlinecolor=GRID_COL, outlinewidth=1,
    )

def spectral_3d(adj):
    n = adj.shape[0]
    if n <= 1:
        return np.random.randn(n, 3) * 0.01
    deg = adj.sum(1)
    d_inv_sqrt = np.where(deg > 0, 1.0 / np.sqrt(np.maximum(deg, 1e-9)), 0.0)
    D = np.diag(d_inv_sqrt)
    L = np.eye(n) - D @ adj @ D
    vals, vecs = np.linalg.eigh(L)
    k = min(4, n)
    pos = vecs[:, 1:k]
    if pos.shape[1] < 3:
        pad = np.random.randn(n, 3 - pos.shape[1]) * 0.05
        pos = np.hstack([pos, pad])
    std = pos.std(0) + 1e-8
    pos = (pos - pos.mean(0)) / std
    return pos[:, :3]

def make_3d_graph_fig(adj, node_colors, title, colorscale="Viridis", showscale=True):
    n   = adj.shape[0]
    pos = spectral_3d(adj)
    ex, ey, ez = [], [], []
    for i in range(n):
        for j in range(i+1, n):
            if adj[i, j] > 0.05:
                ex += [pos[i,0], pos[j,0], None]
                ey += [pos[i,1], pos[j,1], None]
                ez += [pos[i,2], pos[j,2], None]
    edges = go.Scatter3d(x=ex, y=ey, z=ez, mode="lines",
        line=dict(color="rgba(99,102,241,0.30)", width=1.5),
        hoverinfo="none", showlegend=False)
    deg   = adj.sum(1)
    sizes = 6 + 8*(deg - deg.min()) / (deg.max() - deg.min() + 1e-8)
    nodes = go.Scatter3d(
        x=pos[:,0], y=pos[:,1], z=pos[:,2], mode="markers",
        marker=dict(size=sizes.tolist(), color=node_colors, colorscale=colorscale,
                    showscale=showscale, colorbar=_colorbar(),
                    line=dict(color="rgba(0,0,0,0.3)", width=0.5), opacity=0.92),
        hovertext=[f"node {i}" for i in range(n)], hoverinfo="text", showlegend=False)
    scene = dict(bgcolor=DARK_BG,
        xaxis=dict(showgrid=False, showticklabels=False, showline=False, title=dict(text="", font=dict(color=BLK))),
        yaxis=dict(showgrid=False, showticklabels=False, showline=False, title=dict(text="", font=dict(color=BLK))),
        zaxis=dict(showgrid=False, showticklabels=False, showline=False, title=dict(text="", font=dict(color=BLK))),
        camera=dict(eye=dict(x=1.6, y=1.0, z=0.9)))
    fig = go.Figure([edges, nodes])
    fig.update_layout(title=dict(text=title, font=dict(color=BLK, size=13)),
                      scene=scene, height=400, **PLOTLY_LAYOUT)
    return fig

def plot_3d_assignment(s_mat, title, colorscale="Viridis"):
    n_orig, n_super = s_mat.shape
    x_idx, y_idx, z_val = [], [], []
    for i in range(n_orig):
        for j in range(n_super):
            x_idx.append(j); y_idx.append(i); z_val.append(float(s_mat[i,j]))
    fig = go.Figure(go.Scatter3d(x=x_idx, y=y_idx, z=z_val, mode="markers",
        marker=dict(size=4, color=z_val, colorscale=colorscale, showscale=True,
                    colorbar=_colorbar("Weight"), opacity=0.85),
        hovertemplate="<b>Super-node:</b> %{x}<br><b>Orig node:</b> %{y}<br><b>Weight:</b> %{z:.3f}<extra></extra>"))
    fig.update_layout(title=dict(text=title, font=dict(color=BLK, size=13)),
        scene=dict(bgcolor=DARK_BG, xaxis=_ax3d("Super-node"),
                   yaxis=_ax3d("Orig node"), zaxis=_ax3d("Assignment")),
        height=440, **PLOTLY_LAYOUT)
    return fig

def plot_3d_trajectory(history):
    df = pd.DataFrame(history)
    fig = go.Figure(go.Scatter3d(
        x=df["epoch"], y=df["train_acc"], z=df["train_loss"],
        mode="lines+markers",
        marker=dict(size=5, color=df["epoch"], colorscale="Viridis",
                    showscale=True, colorbar=_colorbar("Epoch")),
        line=dict(color="#6366f1", width=2),
        hovertemplate="<b>Epoch:</b> %{x}<br><b>Train Acc:</b> %{y:.1f}%<br><b>Loss:</b> %{z:.4f}<extra></extra>"))
    fig.update_layout(
        title=dict(text="3-D Training Trajectory", font=dict(color=BLK, size=14)),
        scene=dict(bgcolor=DARK_BG, xaxis=_ax3d("Epoch"),
                   yaxis=_ax3d("Train Acc %"), zaxis=_ax3d("Loss")),
        height=520, **PLOTLY_LAYOUT)
    return fig

print("Visualization helpers ready.")


Visualization helpers ready.


---
## 📁 Tab 0 · Dataset Explorer — Actual Graph Structures

Visualize real protein graphs from the PROTEINS dataset.  
Node colour = class label (enzyme / non-enzyme).


In [6]:
import networkx as nx

def pyg_to_nx(data):
    G = nx.Graph()
    G.add_nodes_from(range(data.num_nodes))
    edges = data.edge_index.t().tolist()
    G.add_edges_from(edges)
    return G

def plot_sample_graphs(data_list, n_cols=4, n_rows=2, title="Sample protein graphs"):
    """Plot n_cols*n_rows real graphs from the dataset using spectral layout."""
    n_show = min(n_cols * n_rows, len(data_list))
    fig = make_subplots(rows=n_rows, cols=n_cols,
                        subplot_titles=[f"Graph {i} | n={data_list[i].num_nodes} e={data_list[i].edge_index.shape[1]//2}"
                                        for i in range(n_show)])
    colors_map = {0: "#1D9E75", 1: "#7F77DD"}  # green=non-enzyme, purple=enzyme

    for idx in range(n_show):
        row = idx // n_cols + 1
        col = idx %  n_cols + 1
        d   = data_list[idx]
        G   = pyg_to_nx(d)
        n   = d.num_nodes

        # spectral layout
        adj = to_dense_adj(d.edge_index, max_num_nodes=n)[0].numpy()
        pos2d = spectral_3d(adj)[:, :2]   # use first 2 eigenvectors for 2-D

        label = int(d.y.item())
        node_col = colors_map[label]

        # edges
        ex, ey = [], []
        for u, v in G.edges():
            ex += [pos2d[u,0], pos2d[v,0], None]
            ey += [pos2d[u,1], pos2d[v,1], None]
        fig.add_trace(go.Scatter(x=ex, y=ey, mode="lines",
            line=dict(color="rgba(100,100,100,0.3)", width=0.8),
            hoverinfo="none", showlegend=False), row=row, col=col)

        # node features (feature 0 as size proxy)
        feat0 = d.x[:, 0].numpy() if d.x is not None else np.ones(n)
        sizes = 5 + 6 * (feat0 - feat0.min()) / (feat0.max() - feat0.min() + 1e-8)

        fig.add_trace(go.Scatter(
            x=pos2d[:, 0], y=pos2d[:, 1], mode="markers",
            marker=dict(size=sizes, color=node_col, opacity=0.85,
                        line=dict(color="white", width=0.4)),
            hovertext=[f"node {i}<br>feat: {d.x[i].tolist()}" for i in range(n)] if d.x is not None else None,
            hoverinfo="text", showlegend=False), row=row, col=col)

    fig.update_layout(height=n_rows*260,
                      title=dict(text=title, font=dict(color=BLK, size=14)),
                      **PLOTLY_LAYOUT)
    fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
    fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)

    # colour legend
    for lbl, col, name in [(0,"#1D9E75","Non-enzyme"), (1,"#7F77DD","Enzyme")]:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
            marker=dict(size=10, color=col),
            name=name, showlegend=True))
    fig.update_layout(legend=dict(bgcolor=PANEL_BG, font=dict(color=BLK),
                                  orientation="h", y=1.02, x=0))
    return fig

# ── Enzyme vs Non-enzyme samples ──
enzyme_samples  = [d for d in all_data if int(d.y.item()) == 1][:8]
nonenz_samples  = [d for d in all_data if int(d.y.item()) == 0][:8]

print("Enzyme samples:")
fig_enz = plot_sample_graphs(enzyme_samples, n_cols=4, n_rows=2,
                              title="Enzyme proteins (class = 1)")
fig_enz.show()


Enzyme samples:


In [7]:
print("Non-enzyme samples:")
fig_non = plot_sample_graphs(nonenz_samples, n_cols=4, n_rows=2,
                              title="Non-enzyme proteins (class = 0)")
fig_non.show()


Non-enzyme samples:


In [11]:
# ── Dataset statistics ──
def plot_dataset_stats(data_list):
    node_counts  = [d.num_nodes for d in data_list]
    edge_counts  = [d.edge_index.shape[1]//2 for d in data_list]
    labels       = [int(d.y.item()) for d in data_list]
    degrees      = []
    for d in data_list:
        deg = torch.zeros(d.num_nodes)
        deg.scatter_add_(0, d.edge_index[0], torch.ones(d.edge_index.shape[1]))
        degrees.extend(deg.tolist())

    fig = make_subplots(rows=2, cols=2,
        subplot_titles=("Node count distribution", "Edges vs nodes",
                        "Class balance", "Degree distribution"),
        specs=[[{"type": "xy"}, {"type": "xy"}],
               [{"type": "domain"}, {"type": "xy"}]])

    # Histogram of node counts
    enz_nc  = [node_counts[i] for i,l in enumerate(labels) if l==1]
    nenz_nc = [node_counts[i] for i,l in enumerate(labels) if l==0]
    fig.add_trace(go.Histogram(x=enz_nc,  name="Enzyme",     marker_color="#7F77DD",
                               opacity=0.7, nbinsx=20), row=1, col=1)
    fig.add_trace(go.Histogram(x=nenz_nc, name="Non-enzyme", marker_color="#1D9E75",
                               opacity=0.7, nbinsx=20), row=1, col=1)

    # Scatter edges vs nodes
    ec_enz  = [edge_counts[i] for i,l in enumerate(labels) if l==1]
    ec_nenz = [edge_counts[i] for i,l in enumerate(labels) if l==0]
    fig.add_trace(go.Scatter(x=enz_nc,  y=ec_enz,  mode="markers", name="Enzyme",
                             marker=dict(color="#7F77DD", size=4, opacity=0.6),
                             showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=nenz_nc, y=ec_nenz, mode="markers", name="Non-enzyme",
                             marker=dict(color="#1D9E75", size=4, opacity=0.6),
                             showlegend=False), row=1, col=2)

    # Pie: class balance
    n_enz  = sum(1 for l in labels if l==1)
    n_nenz = sum(1 for l in labels if l==0)
    fig.add_trace(go.Pie(labels=["Enzyme","Non-enzyme"], values=[n_enz, n_nenz],
                         marker_colors=["#7F77DD","#1D9E75"], hole=0.4,
                         showlegend=False), row=2, col=1)

    # Degree distribution
    max_deg = int(max(degrees))
    deg_bins = list(range(min(max_deg+1, 15)))
    deg_hist = [degrees.count(d) for d in deg_bins]
    fig.add_trace(go.Bar(x=deg_bins, y=deg_hist, marker_color="#AFA9EC",
                         showlegend=False), row=2, col=2)

    fig.update_layout(height=640, barmode="overlay",
                      title=dict(text="PROTEINS dataset statistics", font=dict(color=BLK, size=14)),
                      legend=dict(bgcolor=PANEL_BG, font=dict(color=BLK)),
                      **PLOTLY_LAYOUT)
    fig.update_xaxes(**_ax2d("Node count"),  row=1, col=1)
    fig.update_yaxes(**_ax2d("Count"),       row=1, col=1)
    fig.update_xaxes(**_ax2d("Nodes"),       row=1, col=2)
    fig.update_yaxes(**_ax2d("Edges"),       row=1, col=2)
    fig.update_xaxes(**_ax2d("Degree"),      row=2, col=2)
    fig.update_yaxes(**_ax2d("Frequency"),   row=2, col=2)
    return fig

fig_stats = plot_dataset_stats(all_data)
fig_stats.show()


In [12]:
# ── Node feature distributions ──
def plot_node_features(data_list, n_feat=3):
    fig = make_subplots(rows=1, cols=n_feat,
        subplot_titles=[f"Feature {i}" for i in range(n_feat)])
    colors = {0: "#1D9E75", 1: "#7F77DD"}
    for feat_idx in range(n_feat):
        for cls, col in colors.items():
            vals = []
            for d in data_list:
                if int(d.y.item()) == cls and d.x is not None:
                    vals.extend(d.x[:, feat_idx].tolist())
            fig.add_trace(go.Histogram(
                x=vals, name=("Enzyme" if cls==1 else "Non-enzyme"),
                marker_color=col, opacity=0.65, nbinsx=20,
                showlegend=(feat_idx == 0)), row=1, col=feat_idx+1)
    fig.update_layout(barmode="overlay", height=300,
        title=dict(text="Node feature distributions by class", font=dict(color=BLK, size=13)),
        legend=dict(bgcolor=PANEL_BG, font=dict(color=BLK)),
        **PLOTLY_LAYOUT)
    for i in range(n_feat):
        fig.update_xaxes(**_ax2d(f"Feature {i} value"), row=1, col=i+1)
        fig.update_yaxes(**_ax2d("Count"),               row=1, col=i+1)
    return fig

fig_feat = plot_node_features(all_data, n_feat=N_FEAT)
fig_feat.show()


In [13]:
# ── Single graph deep-dive ──
INSPECT_IDX = 0   # ← change this to inspect a different graph

d = all_data[INSPECT_IDX]
adj = to_dense_adj(d.edge_index, max_num_nodes=d.num_nodes)[0].numpy()

print(f"Graph {INSPECT_IDX}:")
print(f"  Class     : {'Enzyme' if int(d.y.item())==1 else 'Non-enzyme'}")
print(f"  Nodes     : {d.num_nodes}")
print(f"  Edges     : {d.edge_index.shape[1]//2}")
print(f"  Features  : {d.x.shape if d.x is not None else 'None'}")
if d.x is not None:
    print(f"  Feature stats:")
    for i in range(d.x.shape[1]):
        v = d.x[:, i]
        print(f"    feat[{i}]  min={v.min():.3f}  max={v.max():.3f}  mean={v.mean():.3f}")

# 3-D view of this single graph
deg   = adj.sum(1)
pos3d = spectral_3d(adj)
fig_single = make_3d_graph_fig(
    adj, node_colors=deg.tolist(),
    title=f"Graph {INSPECT_IDX} — {'Enzyme' if int(d.y.item())==1 else 'Non-enzyme'} | {d.num_nodes} nodes",
    colorscale="Plasma")
fig_single.show()


Graph 0:
  Class     : Non-enzyme
  Nodes     : 42
  Edges     : 81
  Features  : torch.Size([42, 3])
  Feature stats:
    feat[0]  min=0.000  max=1.000  mean=0.524
    feat[1]  min=0.000  max=1.000  mean=0.476
    feat[2]  min=0.000  max=0.000  mean=0.000


---
## 🚀 Tab 1 · Training

Run the cell below to train the DiffPool model.  
Progress prints every 5 epochs; the live chart updates every 10.


In [14]:
def train_one_epoch(model, data, optimizer, max_nodes, device, bs=32):
    model.train()
    perm = torch.randperm(len(data))
    data = [data[i] for i in perm]
    tot_loss = correct = total = 0
    for i in range(0, len(data), bs):
        chunk = data[i:i+bs]
        x, adj, mask, y = dense_batch(chunk, max_nodes, device)
        optimizer.zero_grad()
        out, aux = model(x, adj, mask)
        loss = F.nll_loss(out, y) + 0.5 * aux
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        tot_loss += loss.item() * len(chunk)
        correct  += (out.argmax(-1) == y).sum().item()
        total    += len(chunk)
    return tot_loss / total, correct / total

@torch.no_grad()
def evaluate(model, data, max_nodes, device, bs=32):
    model.eval()
    correct = total = 0
    for i in range(0, len(data), bs):
        chunk = data[i:i+bs]
        x, adj, mask, y = dense_batch(chunk, max_nodes, device)
        out, _ = model(x, adj, mask)
        correct += (out.argmax(-1) == y).sum().item()
        total   += len(chunk)
    return correct / total

print("Training helpers defined.")


Training helpers defined.


In [15]:
from IPython.display import clear_output, display
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = DiffPoolNet(N_FEAT, N_CLS, ACT_MAX, HIDDEN).to(device)
optim  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS, eta_min=1e-5)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {n_params:,} trainable parameters | device: {device}")
print(f"Pool sizes → p1={model.p1}, p2={model.p2}")

history  = {"epoch": [], "train_loss": [], "train_acc": [], "test_acc": []}
best_acc = 0.0
t_start  = time.time()

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_data, optim, ACT_MAX, device, BATCH_SIZE)
    te_acc          = evaluate(model, test_data, ACT_MAX, device, BATCH_SIZE)
    sched.step()

    history["epoch"].append(ep)
    history["train_loss"].append(round(tr_loss, 5))
    history["train_acc"].append(round(tr_acc * 100, 2))
    history["test_acc"].append(round(te_acc * 100, 2))
    best_acc = max(best_acc, te_acc * 100)

    if ep % 5 == 0 or ep == EPOCHS:
        print(f"Ep {ep:>4}/{EPOCHS}  loss={tr_loss:.4f}  "
              f"train={tr_acc*100:.1f}%  test={te_acc*100:.1f}%  "
              f"best={best_acc:.1f}%  lr={sched.get_last_lr()[0]:.2e}  "
              f"({time.time()-t0:.1f}s)")

print(f"\nTraining done in {time.time()-t_start:.0f}s")
if best_acc >= 65:
    print(f"✅ Target reached! Best test accuracy = {best_acc:.1f}%")
else:
    print(f"⚠️  Best test accuracy = {best_acc:.1f}% (below 65% target)")


Model: 56,319 trainable parameters | device: cpu
Pool sizes → p1=25, p2=6
Ep    5/120  loss=1.3235  train=73.0%  test=75.2%  best=77.1%  lr=9.96e-04  (0.8s)
Ep   10/120  loss=1.2026  train=73.7%  test=74.8%  best=77.1%  lr=9.83e-04  (1.0s)
Ep   15/120  loss=1.1133  train=72.5%  test=73.3%  best=77.1%  lr=9.62e-04  (0.9s)
Ep   20/120  loss=1.0325  train=74.6%  test=73.3%  best=77.1%  lr=9.34e-04  (0.8s)
Ep   25/120  loss=0.9982  train=71.3%  test=75.7%  best=77.6%  lr=8.98e-04  (0.8s)
Ep   30/120  loss=0.9429  train=75.6%  test=75.2%  best=77.6%  lr=8.55e-04  (0.8s)
Ep   35/120  loss=0.8868  train=76.3%  test=75.7%  best=77.6%  lr=8.06e-04  (0.8s)
Ep   40/120  loss=0.8682  train=74.6%  test=72.4%  best=77.6%  lr=7.52e-04  (0.8s)
Ep   45/120  loss=0.8372  train=74.3%  test=75.7%  best=77.6%  lr=6.94e-04  (0.8s)
Ep   50/120  loss=0.7871  train=75.1%  test=76.2%  best=77.6%  lr=6.33e-04  (0.8s)
Ep   55/120  loss=0.7778  train=75.1%  test=74.3%  best=77.6%  lr=5.70e-04  (0.8s)
Ep   60/120  

In [16]:
# ── Live training chart ──
df = pd.DataFrame(history)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Training Loss", "Accuracy (%)"))

fig.add_trace(go.Scatter(x=df.epoch, y=df.train_loss, name="Train Loss",
    line=dict(color="#6366f1", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.epoch, y=df.train_acc, name="Train Acc",
    line=dict(color="#10b981", width=2)), row=1, col=2)
fig.add_trace(go.Scatter(x=df.epoch, y=df.test_acc, name="Test Acc",
    line=dict(color="#f59e0b", width=2, dash="dash")), row=1, col=2)
fig.add_hline(y=65, row=1, col=2,
    line=dict(color="#ef4444", dash="dot", width=1.5),
    annotation_text="65% target", annotation_font_color="#ef4444")
fig.update_layout(height=340,
    legend=dict(bgcolor=PANEL_BG, font=dict(color=BLK)), **PLOTLY_LAYOUT)
for ann in fig.layout.annotations:
    ann.font.color = BLK
fig.update_xaxes(**_ax2d("Epoch"))
fig.update_yaxes(**_ax2d())
fig.show()


---
## 🧬 Tab 2 · 3-D Graph Evolution

Shows how DiffPool progressively coarsens a protein graph.  
Change `SAMPLE_IDX` to inspect a different graph from the batch.


In [17]:
# Capture intermediates
model.eval()
sample = train_data[:8]
with torch.no_grad():
    xs, adjs, masks, _ = dense_batch(sample, ACT_MAX, device)
    _, _, inter = model(xs, adjs, masks, return_intermediates=True)

inter_cpu = {k: v.detach().cpu() for k, v in inter.items()}
print(f"Intermediates captured for {xs.shape[0]} graphs.")
print(f"  adj0 shape: {inter_cpu['adj0'].shape}")
print(f"  adj1 shape: {inter_cpu['adj1'].shape}")
print(f"  adj2 shape: {inter_cpu['adj2'].shape}")


Intermediates captured for 8 graphs.
  adj0 shape: torch.Size([8, 100, 100])
  adj1 shape: torch.Size([8, 25, 25])
  adj2 shape: torch.Size([8, 6, 6])


In [18]:
SAMPLE_IDX = 0   # ← change 0–7 to inspect a different graph

adj0 = inter_cpu["adj0"][SAMPLE_IDX].numpy()
adj1 = inter_cpu["adj1"][SAMPLE_IDX].numpy()
adj2 = inter_cpu["adj2"][SAMPLE_IDX].numpy()
s1   = torch.softmax(inter_cpu["s1"][SAMPLE_IDX], dim=-1).numpy()

n0 = int((adj0.sum(1) > 0).sum())

# Filter to only show "active" super-nodes (those that received assignment mass)
active_idx1 = np.where(s1[:n0].sum(0) > 1e-4)[0]
adj1_active = adj1[active_idx1][:, active_idx1]
n1 = len(active_idx1)

# Remap clusters for visualization
mapper1 = {old: new for new, old in enumerate(active_idx1.tolist())}
cluster_assign = [mapper1.get(c, 0) for c in s1[:n0].argmax(-1).tolist()]

# Similarly for Pool 2
active_idx2 = np.where(adj2.sum(1) > 0)[0]
if len(active_idx2) == 0: active_idx2 = np.array([0])
adj2_active = adj2[active_idx2][:, active_idx2]
n2 = len(active_idx2)

print(f"Graph {SAMPLE_IDX}: {n0} nodes → {n1} super-nodes → {n2} super-nodes")
compression = 100*(1 - n2/n0) if n0 > 0 else 0
print(f"Compression: {compression:.0f}%")

fig_orig = make_3d_graph_fig(adj0[:n0,:n0], node_colors=cluster_assign,
    title=f"Original ({n0} nodes) — colour = pool-1 cluster", colorscale="Turbo")
fig_orig.show()


Graph 0: 75 nodes → 25 super-nodes → 6 super-nodes
Compression: 92%


In [19]:
fig_p1 = make_3d_graph_fig(adj1_active, node_colors=list(range(n1)),
    title=f"After Pool-1 ({n1} super-nodes)", colorscale="Viridis")
fig_p1.show()


In [20]:
fig_p2 = make_3d_graph_fig(adj2_active, node_colors=list(range(n2)),
    title=f"After Pool-2 ({n2} super-nodes)", colorscale="Plasma")
fig_p2.show()


---
## 🗺️ Tab 3 · Assignment Maps

Visualizes the soft-assignment matrices **S₁** and **S₂**.  
Each point = (original node, super-node, assignment weight).


In [21]:
ASSIGN_IDX = 0   # ← change 0–7

s1_mat = torch.softmax(inter_cpu["s1"][ASSIGN_IDX], dim=-1).numpy()
s2_mat = torch.softmax(inter_cpu["s2"][ASSIGN_IDX], dim=-1).numpy()
n_show = min(40, s1_mat.shape[0])

# 3-D assignment scatter
fig_s1_3d = plot_3d_assignment(s1_mat[:n_show], "S₁ — Original → Pool-1", "Viridis")
fig_s1_3d.show()


In [22]:
fig_s2_3d = plot_3d_assignment(s2_mat, "S₂ — Pool-1 → Pool-2", "Plasma")
fig_s2_3d.show()


In [23]:
# 2-D heatmaps
fig_hmaps = make_subplots(rows=1, cols=2,
    subplot_titles=("S₁ heatmap — Original → Pool-1", "S₂ heatmap — Pool-1 → Pool-2"))
fig_hmaps.add_trace(go.Heatmap(z=s1_mat[:n_show], colorscale="Viridis",
    showscale=True, colorbar=_colorbar()), row=1, col=1)
fig_hmaps.add_trace(go.Heatmap(z=s2_mat, colorscale="Plasma",
    showscale=True, colorbar=_colorbar()), row=1, col=2)
fig_hmaps.update_layout(height=360,
    title=dict(text="2-D assignment heatmaps", font=dict(color=BLK, size=13)),
    **PLOTLY_LAYOUT)
fig_hmaps.update_xaxes(**_ax2d("Super-node"), row=1, col=1)
fig_hmaps.update_yaxes(**_ax2d("Orig node"),  row=1, col=1)
fig_hmaps.update_xaxes(**_ax2d("Super-node"), row=1, col=2)
fig_hmaps.update_yaxes(**_ax2d("Pool-1 node"),row=1, col=2)
fig_hmaps.show()


---
## 📊 Tab 4 · Analytics

Full training analytics: KPI summary, accuracy curves, 3-D trajectory, loss curve, raw log.


In [24]:
df = pd.DataFrame(history)

# KPI summary
print("┌──────────────────────────────────────┐")
print("│  Training summary                     │")
print("├──────────────────────────────────────┤")
kpis = [
    ("Best test accuracy",  f"{best_acc:.1f}%  {'✅' if best_acc>=65 else '❌'}"),
    ("Final test accuracy", f"{history['test_acc'][-1]:.1f}%"),
    ("Final train loss",    f"{history['train_loss'][-1]:.4f}"),
    ("Epochs completed",    str(EPOCHS)),
]
for k, v in kpis:
    print(f"│  {k:<22} {v:<14} │")
print("└──────────────────────────────────────┘")


┌──────────────────────────────────────┐
│  Training summary                     │
├──────────────────────────────────────┤
│  Best test accuracy     79.0%  ✅       │
│  Final test accuracy    74.3%          │
│  Final train loss       0.6458         │
│  Epochs completed       120            │
└──────────────────────────────────────┘


In [25]:
# Accuracy curves
fig_acc = go.Figure()
fig_acc.add_trace(go.Scatter(x=df.epoch, y=df.train_acc, name="Train Acc",
    fill="tozeroy", fillcolor="rgba(99,102,241,0.12)",
    line=dict(color="#6366f1", width=2)))
fig_acc.add_trace(go.Scatter(x=df.epoch, y=df.test_acc, name="Test Acc",
    line=dict(color="#f59e0b", width=2, dash="dash")))
fig_acc.add_hline(y=65, line=dict(color="#ef4444", dash="dot"),
    annotation_text="65% target", annotation_font_color="#ef4444")
fig_acc.update_layout(height=360,
    title=dict(text="Accuracy curves", font=dict(color=BLK, size=14)),
    xaxis=_ax2d("Epoch"), yaxis=_ax2d("Accuracy (%)"),
    legend=dict(bgcolor=PANEL_BG, font=dict(color=BLK),
                bordercolor=GRID_COL, borderwidth=1),
    **PLOTLY_LAYOUT)
fig_acc.show()


In [26]:
# 3-D training trajectory
plot_3d_trajectory(history).show()


In [27]:
# Loss curve
fig_loss = go.Figure(go.Scatter(x=df.epoch, y=df.train_loss, name="Train Loss",
    fill="tozeroy", fillcolor="rgba(168,85,247,0.12)",
    line=dict(color="#a855f7", width=2)))
fig_loss.update_layout(height=300,
    title=dict(text="Training loss", font=dict(color=BLK, size=14)),
    xaxis=_ax2d("Epoch"), yaxis=_ax2d("Loss"), **PLOTLY_LAYOUT)
fig_loss.show()


In [28]:
# Raw training log
print(df.to_string(index=False,
    formatters={"train_loss": "{:.5f}".format,
                "train_acc":  "{:.2f}".format,
                "test_acc":   "{:.2f}".format}))


 epoch train_loss train_acc test_acc
     1    1.61155     70.33    43.33
     2    1.47487     72.37    72.38
     3    1.42746     69.86    75.71
     4    1.35368     71.65    77.14
     5    1.32345     72.97    75.24
     6    1.28355     71.53    76.19
     7    1.24535     74.40    70.00
     8    1.24317     72.73    75.24
     9    1.21481     73.56    76.19
    10    1.20263     73.68    74.76
    11    1.17756     73.56    74.76
    12    1.17147     71.17    74.76
    13    1.14002     72.25    73.81
    14    1.11993     74.88    73.33
    15    1.11334     72.49    73.33
    16    1.09012     73.56    76.19
    17    1.06309     72.85    72.86
    18    1.05530     73.80    72.38
    19    1.04485     74.04    76.19
    20    1.03252     74.64    73.33
    21    1.01500     73.80    75.71
    22    1.01122     73.33    77.62
    23    0.99696     74.52    76.19
    24    0.97957     74.16    73.81
    25    0.99824     71.29    75.71
    26    0.96784     73.92    72.38
 